# 6. Pipelines and mixed columns

## Small idea: one fitted object should remember the whole recipe

A `ColumnTransformer` applies different transformations to numerical, categorical, and
text columns. A `Pipeline` orders dependent steps. Together they keep training and future
transformations consistent and reduce leakage.

**Learning goals**

- build separate pipelines for numerical and categorical features;
- combine them with `ColumnTransformer`;
- fit once on training data and reuse on validation/test data;
- inspect output names and shapes;
- prepare the exact structure that Stage 6 estimators will consume.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

train = pd.DataFrame({
    "age": [21.0, 34.0, np.nan, 28.0, 25.0, 31.0],
    "tokens": [80, 210, 130, 95, 160, 115],
    "task_type": ["free", "picture", "free", "social", None, "picture"],
    "proficiency": ["A2", "B2", "B1", "A2", "B1", "B2"],
})

validation = pd.DataFrame({
    "age": [26.0, np.nan],
    "tokens": [105, 190],
    "task_type": ["interview", "free"],
    "proficiency": ["B1", "C1"],
})

## 1. Define column roles explicitly

In [ ]:
numeric_features = ["age", "tokens"]
categorical_features = ["task_type", "proficiency"]

assert set(numeric_features).isdisjoint(categorical_features)
assert set(numeric_features + categorical_features) == set(train.columns)

## 2. Build one pipeline per feature type

In [ ]:
numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

## 3. Combine the branches

In [ ]:
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

X_train = preprocessor.fit_transform(train)
X_validation = preprocessor.transform(validation)

print("Training shape:", X_train.shape)
print("Validation shape:", X_validation.shape)
assert X_train.shape[1] == X_validation.shape[1]

In [ ]:
feature_names = preprocessor.get_feature_names_out()
pd.DataFrame(X_train, columns=feature_names).round(2)

The validation-only categories `interview` and `C1` do not create new columns. The output
schema is determined by training data and reused unchanged.

## 4. Add a feature-selection step

In [ ]:
feature_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("remove_constant", VarianceThreshold(threshold=0.0)),
])

X_train_selected = feature_pipeline.fit_transform(train)
X_validation_selected = feature_pipeline.transform(validation)

support = feature_pipeline.named_steps["remove_constant"].get_support()
selected_names = feature_pipeline.named_steps["preprocess"].get_feature_names_out()[support]

print("Selected names:", selected_names.tolist())
print("Selected shapes:", X_train_selected.shape, X_validation_selected.shape)

## 5. The Stage 6 bridge

In the next stage, an estimator becomes the final pipeline step:

```python
model_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", estimator),
])
```

Cross-validation will then refit every learned preprocessing step inside each training
fold. The estimator and its evaluation belong to Stage 6; the feature contract belongs here.

## Pipeline checklist

- Column lists are explicit and target-free.
- Every learned transformer is fitted on training data only.
- Unknown categories have a defined policy.
- Validation/test matrices have the same feature width as training.
- Feature names and transformation choices are documented.
- Random seeds and split groups are recorded.

## Tiny checkpoint

Add a `response_seconds` numerical column and a `source` categorical column. Update the
column lists, rerun the pipeline, and inspect the new output names.